# 既在范围里，又留下垂直残差

单声道只能输出(c,c)，先比较候选，再配方覆盖所有c。平面例则用上一章已求出的法向量：目标(2,3,6)减去c(−1,−1,1)，用z=x+y确定c。最后才用内部正交基核对同一点，不凭空交出两支漂亮方向。

从新内核按顺序运行；没有滑块时可直接修改参数。计算单元用于核对前面的推理，参考结果放在相应尝试之后。


In [ ]:
# 让图中的中文正常显示：先找本机已有的中文字体；找不到时（例如 Colab）下载思源黑体。
import pathlib
import urllib.request
import matplotlib.pyplot as plt
from matplotlib import font_manager

def use_chinese_font():
    candidates = ['Noto Sans CJK SC', 'Noto Sans SC', 'Source Han Sans SC', 'PingFang SC',
                  'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'WenQuanYi Zen Hei']
    installed = {font.name for font in font_manager.fontManager.ttflist}
    name = next((n for n in candidates if n in installed), None)
    if name is None:
        path = pathlib.Path.home() / '.cache' / 'linear-algebra-fonts' / 'NotoSansSC-Regular.otf'
        try:
            if not path.exists():
                path.parent.mkdir(parents=True, exist_ok=True)
                urllib.request.urlretrieve(
                    'https://github.com/notofonts/noto-cjk/raw/main/Sans/SubsetOTF/SC/'
                    'NotoSansSC-Regular.otf', path)
            font_manager.fontManager.addfont(str(path))
            name = font_manager.FontProperties(fname=str(path)).get_name()
        except Exception as error:
            print('没有找到中文字体，图中汉字可能显示为方框：', error)
            return
    plt.rcParams['font.family'] = [name, 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False

use_chinese_font()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def check_prediction(value, expected, label):
    if value is None:
        print(label + '：尚未填写，请先在纸上计算，再替换 None。')
        return None
    try:
        proposed = np.asarray(value, dtype=float)
    except (TypeError, ValueError, OverflowError):
        print(label + '：请填写数值、数值列表或数组。')
        return False
    reference = np.asarray(expected, dtype=float)
    if proposed.shape != reference.shape:
        print(label + '：尺寸应为 ' + str(reference.shape) + '，请检查行列。')
        return False
    if not np.all(np.isfinite(proposed)):
        print(label + '：请检查是否除以零或产生了非有限数。')
        return False
    correct = bool(np.allclose(proposed, reference))
    print(label + ('：核对正确。' if correct else '：尚不一致，请回到方程或内积检查。'))
    return correct


## 比三个候选，再覆盖所有系数

目标(3,1)，先算 c=1、2、3 的平方误差，再展开任意c并配方；下一格用曲线核对，不把曲线最低点替代证明。


In [ ]:
u = np.array([1.,1.])
b = np.array([3.,1.])
coefficient = (u @ b)/(u @ u)
p = coefficient*u
r = b-p
assert np.allclose(u @ r,0)
c = np.linspace(-1,4,301)
errors = ((b[:,None]-u[:,None]*c)**2).sum(axis=0)
fig,axes = plt.subplots(1,2,figsize=(10,4.5),dpi=150)
axes[0].plot(c,errors,color='#2878b5')
axes[0].scatter(coefficient,r @ r,color='#c53351',zorder=4)
axes[0].set(xlabel='两边共同输出的数 c',ylabel='误差平方和',title='哪个 c 误差最小')
axes[1].plot([-1,4],[-1,4],color='#2878b5',label='能输出的 (c,c)')
axes[1].scatter(*b,color='#c53351',label='目标 (3,1)')
axes[1].scatter(*p,color='#359969',label='最近输出 (2,2)')
axes[1].plot([p[0],b[0]],[p[1],b[1]],'--',color='#d97922',label='残差，与直线垂直')
axes[1].set(xlim=(-1,4),ylim=(-1,4),aspect='equal',xlabel='左声道 L',ylabel='右声道 R',title='最近的可达点')
axes[1].legend()
for ax in axes: ax.grid(alpha=.2)
fig.tight_layout()
plt.show()


## 直接改变候选 c

预测误差，然后修改 candidate。


In [ ]:
candidate = 1.5
error = b-candidate*u
print('候选输出:',candidate*u,'误差平方:',error @ error)
assert error @ error >= r @ r - 1e-12


## 沿已知的垂直方向修正

目标(2,3,6)，法向量n=(−1,−1,1)。设最近点为目标减cn，要求第三项等于前两项之和，求c，再把最近点填入下一格。随后代码从平面条件与垂直条件构造内部正交基，核对相同点。


In [ ]:
target_plane = np.array([2.,3.,6.])
predicted_projection = None
check_prediction(predicted_projection, [7/3,10/3,17/3], '平面投影预测')
normal_plane = np.array([-1.,-1.,1.])
normal_amount = (normal_plane @ target_plane) / (normal_plane @ normal_plane)
projection_from_normal = target_plane-normal_amount*normal_plane
first_plane = np.array([1.,0.,1.])
# 平面条件 c=a+b，垂直条件 a+c=0，取 a=-1。
second_plane = np.array([-1.,2.,1.])
Qplane = np.column_stack((first_plane/np.linalg.norm(first_plane),
                         second_plane/np.linalg.norm(second_plane)))
assert np.allclose(Qplane.T @ Qplane, np.eye(2))
coordinates_plane = Qplane.T @ target_plane
projected_plane = Qplane @ coordinates_plane
residual_plane = target_plane - projected_plane
print('坐标:', coordinates_plane, '投影:', projected_plane)
print('残差:', residual_plane)
assert np.allclose(projected_plane, [7/3,10/3,17/3])
assert np.allclose(Qplane.T @ residual_plane, 0)
assert np.isclose(projected_plane[2], projected_plane[0]+projected_plane[1])
Pplane = Qplane @ Qplane.T
assert np.allclose(Pplane @ Pplane, Pplane)
assert np.allclose(Pplane.T, Pplane)
assert not np.allclose(Pplane, np.eye(3))

assert np.allclose(projected_plane, projection_from_normal)


## 比较候选，并核对勾股拆分

候选 (2,3,5) 只改总额。先预测它的误差平方与最近点的误差平方。再找另一个平面内候选：不必猜它最优，但必须解释为什么误差不会更小。


In [ ]:
for alternative in [np.array([2.,3.,5.]), np.array([1.,4.,5.])]:
    error2 = np.linalg.norm(target_plane-alternative)**2
    best2 = np.linalg.norm(residual_plane)**2
    extra2 = np.linalg.norm(projected_plane-alternative)**2
    print('候选:', alternative, '误差:', error2, '=', best2, '+', extra2)
    assert np.isclose(error2, best2+extra2)
    assert error2 >= best2-1e-12
assert np.isclose(np.linalg.norm(residual_plane)**2, 1/3)


<details><summary>完成上面的尝试后核对</summary>
最近点为 (7/3,10/3,17/3)，残差为 (-1,-1,1)/3，平方误差 1/3。只修第三项误差为 1。正交残差加上任意平面内的额外变化，平方长度相加，所以不可能更好。Q是三行两列，QQ^T是投影，不是单位矩阵。
</details>


## 投影不一致的报表

三个字段同等对待时，为什么不只修改总额？


In [ ]:
report = np.array([120.,80.,210.])
normal = np.array([-1.,-1.,1.])
residual = normal*(normal @ report)/(normal @ normal)
consistent = report-residual
print('最近一致报表:',consistent,'残差:',residual)
assert np.allclose(consistent[2],consistent[0]+consistent[1])
C = np.array([[1.,0.],[0.,1.],[1.,1.]])
assert np.allclose(C.T @ residual,0)


## 用同一条件重新找点

从法向修正、内部正交基两种求法得到同一点，说明残差正交为何排除所有更优候选。只许修报表第三项时，解释原来的等权最优点为什么不再可用。
